# Dim Calendar Table

Creates a calendar dimension table `retail_q.retail_gold.dim_calendar` populated from user-specified start and end dates. Includes standard calendar attributes: year, quarter, month, week, day, and weekend/end-of-month flags.

In [0]:
dbutils.widgets.text(name="start_date", defaultValue="2024-01-01", label="Start Date")
dbutils.widgets.text(name="end_date", defaultValue="2025-12-31", label="End Date")

print("start_date:", dbutils.widgets.get("start_date"))
print("end_date:", dbutils.widgets.get("end_date"))

In [0]:
%sql
CREATE OR REPLACE TABLE retail_q.retail_gold.dim_calendar AS
SELECT
  date_col                                                       AS date,
  YEAR(date_col)                                                 AS year,
  QUARTER(date_col)                                              AS quarter,
  CONCAT('Q', CAST(QUARTER(date_col) AS STRING))                 AS quarter_name,
  MONTH(date_col)                                                AS month,
  DATE_FORMAT(date_col, 'MMMM')                                  AS month_name,
  DATE_FORMAT(date_col, 'MMM')                                   AS month_short_name,
  WEEKOFYEAR(date_col)                                          AS week_of_year,
  CEIL(DAYOFMONTH(date_col) / 7.0)                               AS week_of_month,
  DAYOFYEAR(date_col)                                           AS day_of_year,
  DAYOFMONTH(date_col)                                          AS day_of_month,
  DAYOFWEEK(date_col)                                            AS day_of_week,
  DATE_FORMAT(date_col, 'EEEE')                                  AS day_of_week_name,
  DATE_FORMAT(date_col, 'EEE')                                   AS day_of_week_short_name,
  CASE WHEN DAYOFWEEK(date_col) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  (date_col = LAST_DAY(date_col))                                AS is_end_of_month,
  (DAYOFMONTH(date_col) = 1)                                     AS is_start_of_month
FROM (
  SELECT EXPLODE(SEQUENCE(TO_DATE(:start_date), TO_DATE(:end_date), INTERVAL 1 DAY)) AS date_col
);

In [0]:
%sql
SELECT count(*) AS row_count, min(date) AS min_date, max(date) AS max_date
FROM retail_q.retail_gold.dim_calendar;